# Workloads de Peopleware
Generador independiente de `archivados/bases_plantillas/archivos/reporte_workloads.xlsx`, hoja **Workloads**, tabla **t_workloads**. Incluye cargas con `planningUnitId` conocido, aunque no tengan forecast. El UUID identifica cada carga; unidad y actividad vienen de su configuración.

El período inicial es **28/09/2026 a 25/10/2026**, inclusive. El resultado usa siempre intervalos de **30 minutos**, sin modificar Peopleware. Se conserva `utc_offset_minutos` para distinguir horas repetidas y revisar cambios de hora; se elimina la columna constante `intervalo_minutos`.

La API nueva entrega las variantes de forecast con `startTime` inclusivo y `endTime` exclusivo. Cada `forecast_<variante>` conserva su volumen `offered` y cada `tmo_<variante>` su tiempo medio en segundos. Son alternativas: no se suman entre sí. Las cinco variantes base mantienen columnas estables; nuevas variantes recibidas se incorporan. Ausente o `null` queda vacío; cero sigue siendo cero.

In [1]:
from datetime import date
from pathlib import Path
import polars as pl
from IPython.display import display
from analytics.excel_reportes import TablaDatos, generar_libro

actual = Path.cwd().resolve()
BASE_DIR = next((p for p in (actual, *actual.parents)
                 if (p / "db_conn.py").is_file() and (p / "pyproject.toml").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Abrir el notebook dentro del proyecto.")
import sys
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from analytics.datos_reportes import cargar_workloads, formatos_workloads
DESDE = date(2026, 9, 28)
HASTA = date(2026, 10, 25)
RUTA_EXCEL = BASE_DIR / "archivados" / "bases_plantillas" / "archivos" / "reporte_workloads.xlsx"

Para 15 → 30 minutos se suman los dos volúmenes completos y se pondera el TMO con el volumen de su propia variante, o con contestadas en el histórico. Sin pesos positivos, con pesos negativos o bloques incompletos, el TMO ponderado queda vacío. Una respuesta nativa de 30 minutos conserva el TMO recibido.

El requerido se obtiene de legacy para la actividad asociada, verificando pertenencia y vigencia. Se promedia por tiempo, conservando horas de dotación. Se evita repetir requerido entre cargas de la misma actividad o entre una actividad agrupada y su hija.

El histórico conserva sólo contestadas y TMO. Su curva debe tener una asociación inequívoca: etiqueta del UUID o nombre exacto normalizado de una curva de exportación automática. Los IDs de colas no se interpretan como IDs de curvas.

La grilla se construye en UTC y se convierte a la zona del Workload. Las series legacy de días con cambio de hora se omiten y se alertan porque sus arrays diarios no permiten reconstruir la transición con seguridad. Series de una hora o un día no se reparten artificialmente.

In [2]:
print("Consultando Workloads y sus series...")
resultado = cargar_workloads(DESDE, HASTA, BASE_DIR)
df = resultado.df
df_asociaciones = resultado.asociaciones
df_cobertura = resultado.cobertura
df_cobertura_diaria = resultado.cobertura_diaria
df_cobertura_fuentes = resultado.cobertura_fuentes
df_alertas = resultado.alertas
df_consultas_sin_datos = resultado.consultas_sin_datos
print(f"Workloads: {df['workload_id'].n_unique()} cargas; {df.height:,} filas; {df.width} columnas.")
display(resultado.perfil)
display(df_asociaciones)
display(df_cobertura)
display(df_alertas)

Consultando Workloads y sus series...


Workloads: 30 cargas; 40,320 filas; 28 columnas.


columna,tipo
str,str
"""fecha""","""Date"""
"""intervalo""","""Time"""
"""utc_offset_minutos""","""Int64"""
"""planning_unit_id""","""Int64"""
"""unidad_planificacion""","""String"""
…,…
"""forecast_adjustments""","""Float64"""
"""tmo_adjustments""","""Float64"""
"""forecast_operational""","""Float64"""


workload_id,workload,canal,zona_horaria,planning_unit_id,unidad_planificacion,activity_id,actividad,raster_configurado,curve_id,estado_historico,estado_requerido
str,str,str,str,i64,str,i64,str,i64,i64,str,str
"""d68d6e9e-439a-42b4-ba84-38209c…","""Bupa Seguros""","""calls""","""America/Santiago""",1036,"""Bupa Seguros""",70766,"""Bupa Seguros""",900,1031,"""exportacion_automatica_nombre_…","""actividad_asociada"""
"""4c4db074-dc55-43d3-b91f-878ce2…","""Caec Outbound Hábil""","""calls""","""America/Santiago""",1069,"""Caec Outbound Hábil""",72036,"""Caec Outbound Hábil""",900,null,"""sin_curva_historica_identifica…","""actividad_asociada"""
"""35dd3579-97fe-45e2-b5db-ed86b2…","""CEAC completo""","""calls""","""America/Santiago""",1065,"""Ceac agrupado""",70847,"""Ceac agrupado""",900,1063,"""exportacion_automatica_nombre_…","""actividad_asociada"""
"""0a68d7ea-2505-43c4-891e-e12c4f…","""Ceac Grupo 2""","""calls""","""America/Santiago""",1065,"""Ceac agrupado""",69858,"""Ceac Grupo 2""",900,null,"""sin_curva_historica_identifica…","""actividad_incluida_en_otro_wor…"
"""0b7f75ad-494f-46b5-b84d-55b7fd…","""Clinica Pasteur""","""calls""","""America/Santiago""",1080,"""Clinica Pasteur""",80042,"""Clinica Pasteur""",900,1066,"""exportacion_automatica_nombre_…","""actividad_asociada"""
…,…,…,…,…,…,…,…,…,…,…,…
"""a8038b70-787d-40d8-975c-ccd142…","""RS Adulto Mayor""","""calls""","""America/Santiago""",1028,"""RS IN Adulto Mayor""",70754,"""RS IN Adulto Mayor""",900,1034,"""exportacion_automatica_nombre_…","""actividad_asociada"""
"""eb8f36a4-1364-4409-be40-ff0c37…","""RS Examenes N1""","""calls""","""America/Santiago""",1026,"""RS IN Examenes""",70721,"""RS IN Examenes""",900,1032,"""exportacion_automatica_nombre_…","""actividad_asociada"""
"""7ff546aa-b431-44c4-8a2b-d1de68…","""RS USA GES""","""calls""","""America/Santiago""",1027,"""RS IN USA""",70753,"""RS IN USA""",900,1048,"""exportacion_automatica_nombre_…","""actividad_asociada"""


columna,valores,vacios,ceros,no_cero
str,i64,i64,i64,i64
"""llamadas_historicas""",11808,28512,6965,4843
"""contestadas_historicas""",11568,28752,6810,4758
"""tmo_historico""",4758,35562,0,4758
"""forecast_raw_auto""",20576,19744,14796,5780
"""tmo_raw_auto""",5780,34540,14,5766
…,…,…,…,…
"""forecast_adjustments""",2544,37776,1664,880
"""tmo_adjustments""",0,40320,0,0
"""forecast_operational""",28368,11952,18854,9514


workload_id,motivo,detalle
str,str,str
"""834a9168-2ba4-4e2a-9ef0-f9c067…","""sin_unidad_asociada""","""Ceac grupo 1"""
"""267cbabf-fd23-433f-ac78-9cb58c…","""sin_unidad_asociada""","""Prueba OUT"""
"""4c4db074-dc55-43d3-b91f-878ce2…","""sin_curva_historica_identifica…","""Caec Outbound Hábil"""
"""0a68d7ea-2505-43c4-891e-e12c4f…","""sin_curva_historica_identifica…","""Ceac Grupo 2"""
"""4f8d1b69-1044-481b-9157-b9bbed…","""sin_curva_historica_identifica…","""Cruz Verde"""
…,…,…
"""db2bf683-6b15-4686-ba20-0285ae…","""bloques_forecast_incompletos_a…","""auto: 1 bloques; volumen conoc…"
"""fb9da768-4b0f-4f7c-8ec7-c49f37…","""bloques_forecast_incompletos_a…","""rawAuto: 1 bloques; volumen co…"
"""fb9da768-4b0f-4f7c-8ec7-c49f37…","""bloques_forecast_incompletos_a…","""auto: 1 bloques; volumen conoc…"


Las comprobaciones validan claves por UUID, fecha, hora y offset; grilla completa; conservación de volúmenes y horas de requerido; y ausencia de requerido duplicado. Los diagnósticos quedan en el notebook, sin añadir hojas al Excel. `df_cobertura_fuentes` permite revisar el comienzo real de cada serie y su marca `outdated`.

In [3]:
print("Escribiendo y validando el Excel...")
generar_libro([TablaDatos("t_workloads", "Workloads", df, formatos_workloads(df))], RUTA_EXCEL)
print(f"Generado: {RUTA_EXCEL.name}; {df.height:,} filas.")

Escribiendo y validando el Excel...


Generado: reporte_workloads.xlsx; 40,320 filas.
